# Local protein structures — render any PDB without AlphaFold

This notebook shows how to add 3D protein structure viewers to a TMAP visualization
when **you already have the structures on disk** — no AlphaFold lookup, no UniProt
required. Structure data comes from your files or URLs; the browser still loads
3Dmol.js from its CDN for the viewer.

The recommended API is `TmapViz.add_3d_structure_files`: pass local paths,
then `write_static()` copies the sidecar files into the output bundle.

Two transports:

| Mode | When to use | What gets stored | Per-point cost in HTML |
|---|---|---|---|
| `add_3d_structure_files(paths)` | Local PDB/CIF files, static bundle | Relative URLs + copied sidecars | ~50 bytes per row |
| `add_3d_structures(urls)` | Already-hosted URLs or hand-managed sidecars | Just the URL/path string | ~50 bytes per row |
| `add_3d_structures(texts, source="text")` | Tiny demos where raw PDB text is acceptable | Full PDB/CIF text gzip+base64 in HTML | KB–MB per row |

The walkthrough builds a realistic project folder, fits a tiny TMAP from
sequence-derived features, and renders file-backed plus text-backed modes for comparison.


## What a typical input folder looks like

Imagine you have a small set of proteins of interest. On disk you'd typically have:

```
my_project/
├── proteins.csv           # id, sequence, name
└── pdbs/                  # one structure file per id
    ├── 1CRN.pdb
    ├── 1UBQ.pdb
    ├── 6LYZ.pdb
    └── ...
```

The PDB files might come from your own AlphaFold runs, downloaded from RCSB,
generated by ESMFold, modeled by Rosetta — TMAP doesn't care. Whatever produced
`<id>.pdb` is upstream of this visualization.

The cell below builds that exact layout for the notebook: it downloads ~10 small
proteins from RCSB once (cached on disk) and writes a small `proteins.csv`.


In [ ]:
from __future__ import annotations
import urllib.request
from pathlib import Path
import pandas as pd

PROJECT_DIR = Path("local_proteins_demo").resolve()
PDB_DIR = PROJECT_DIR / "pdbs"
PDB_DIR.mkdir(parents=True, exist_ok=True)

# (pdb_id, friendly name)
PROTEINS = [
    ("1CRN", "Crambin"),
    ("1UBQ", "Ubiquitin"),
    ("6LYZ", "Lysozyme"),
    ("2JOF", "Trp-cage"),
    ("1MBN", "Myoglobin"),
    ("1HHO", "Hemoglobin alpha"),
    ("1A3N", "Hemoglobin tetramer"),
    ("1IGT", "IgG2a antibody"),
    ("4HHB", "Hemoglobin"),
    ("1BNA", "B-DNA dodecamer"),
]

def fetch_one(pdb_id: str) -> Path:
    out = PDB_DIR / f"{pdb_id}.pdb"
    if not out.exists():
        url = f"https://files.rcsb.org/download/{pdb_id}.pdb"
        with urllib.request.urlopen(url, timeout=30) as r:
            out.write_bytes(r.read())
    return out

paths = [fetch_one(pid) for pid, _ in PROTEINS]
print(f"Wrote {len(paths)} PDBs to {PDB_DIR.relative_to(Path.cwd())}")
print(f"Total disk size: {sum(p.stat().st_size for p in paths) / 1024:.0f} KB")


Wrote 10 PDBs to local_proteins_demo/pdbs
Total disk size: 3392 KB


In [ ]:
# Pretend you already had a CSV; here we synthesize one for the demo.
df = pd.DataFrame({
    "id":   [pid for pid, _ in PROTEINS],
    "name": [name for _, name in PROTEINS],
})
csv_path = PROJECT_DIR / "proteins.csv"
df.to_csv(csv_path, index=False)
print(f"Wrote {csv_path.relative_to(Path.cwd())}")
df


Wrote local_proteins_demo/proteins.csv


,id,name
0,1CRN,Crambin
1,1UBQ,Ubiquitin
2,6LYZ,Lysozyme
3,2JOF,Trp-cage
4,1MBN,Myoglobin
5,1HHO,Hemoglobin alpha
6,1A3N,Hemoglobin tetramer
7,1IGT,IgG2a antibody
8,4HHB,Hemoglobin
9,1BNA,B-DNA dodecamer


In [ ]:
# Confirm the folder layout matches what the docs describe.
for p in sorted(PROJECT_DIR.rglob("*")):
    rel = p.relative_to(PROJECT_DIR)
    indent = "  " * (len(rel.parts) - 1)
    suffix = "/" if p.is_dir() else f"  ({p.stat().st_size // 1024 or 1} KB)"
    print(f"{indent}{rel.name}{suffix}")


pdbs/
  1A3N.pdb  (450 KB)
  1BNA.pdb  (74 KB)
  1CRN.pdb  (48 KB)
  1HHO.pdb  (238 KB)
  1IGT.pdb  (1116 KB)
  1MBN.pdb  (143 KB)
  1UBQ.pdb  (76 KB)
  2JOF.pdb  (648 KB)
  4HHB.pdb  (462 KB)
  6LYZ.pdb  (131 KB)
proteins.csv  (1 KB)
viz_inline/
viz_url/


## Build a small TMAP from per-protein features

For the demo we derive a tiny feature vector per protein from physicochemical
properties (no external models needed) and project to 2D with PCA. In a real
project you'd substitute pre-computed embeddings (ESM, ProtTrans, …) or use
`TMAP(...).fit_transform` on a larger dataset.

The point of this notebook is the structure viewer wiring, not the projection,
so any 2D layout will do.


In [ ]:
import numpy as np
from tmap.utils.proteins import read_pdb, AVAILABLE_SEQUENCE_PROPERTIES, sequence_properties

# Pull the amino acid sequence out of each PDB (skips DNA-only entries gracefully)
records = [read_pdb(p) for p in paths]
sequences = [r["sequence"] for r in records]

# Replace empty sequences (e.g. 1BNA is DNA only) with a placeholder so we still
# get a row in the visualization.
sequences = [s if s else "A" for s in sequences]

props = sequence_properties(sequences)
feature_matrix = np.column_stack([props[k] for k in AVAILABLE_SEQUENCE_PROPERTIES])
feature_matrix = np.nan_to_num(feature_matrix, nan=0.0)

# z-score then PCA to 2D
X = (feature_matrix - feature_matrix.mean(0)) / (feature_matrix.std(0) + 1e-9)
U, S, Vt = np.linalg.svd(X, full_matrices=False)
xy = U[:, :2] * S[:2]
print("2D coords:"); print(xy.round(2))


2D coords:
[[ 1.73 -1.25]
 [-0.15  1.96]
 [-1.61  0.23]
 [-0.73  0.38]
 [-1.24  2.49]
 [-0.23  0.71]
 [-0.87  0.03]
 [-2.54 -4.01]
 [-0.84  0.03]
 [ 6.47 -0.58]]


## Render the visualization in URL mode (sidecar directory)

This is the path you want for thousands of structures. The HTML stores only the
relative URL string (`pdbs/1CRN.pdb`) per row; the browser fetches each file
lazily when the user pins that point.

Note the call to `viz.write_static(out_dir, ...)` — it writes `index.html`
plus a few JSON/binary support files into `out_dir`. We then copy our `pdbs/`
folder next to it so the relative URLs resolve.


In [ ]:
from tmap.visualization import TmapViz

OUT_URL = PROJECT_DIR / "viz_url"
OUT_URL.mkdir(exist_ok=True)

viz = TmapViz()
viz.title = "Local proteins — file-backed mode"
viz.point_size = 18.0
viz.set_points(xy[:, 0], xy[:, 1])
viz.add_label("PDB ID", df["id"].tolist())
viz.add_label("Name", df["name"].tolist())
viz.add_3d_structure_files(paths, fmt="pdb")

viz.write_static(OUT_URL)

html_size = (OUT_URL / "index.html").stat().st_size / 1024
print(f"index.html: {html_size:.0f} KB  (no PDBs embedded)")


index.html: 597 KB  (no PDBs embedded)


In [ ]:
# Inspect the output bundle.
for p in sorted(OUT_URL.rglob("*"))[:30]:
    rel = p.relative_to(OUT_URL)
    indent = "  " * (len(rel.parts) - 1)
    suffix = "/" if p.is_dir() else f"  ({p.stat().st_size // 1024 or 1} KB)"
    print(f"{indent}{rel.name}{suffix}")


col_Name.bin  (1 KB)
col_PDB_ID.bin  (1 KB)
col_structure.bin  (1 KB)
coords.bin  (1 KB)
index.html  (597 KB)
metadata.json  (1 KB)
pdbs/
  1A3N.pdb  (450 KB)
  1BNA.pdb  (74 KB)
  1CRN.pdb  (48 KB)
  1HHO.pdb  (238 KB)
  1IGT.pdb  (1116 KB)
  1MBN.pdb  (143 KB)
  1UBQ.pdb  (76 KB)
  2JOF.pdb  (648 KB)
  4HHB.pdb  (462 KB)
  6LYZ.pdb  (131 KB)


### How to view

The visualization is a static folder; open it locally:

```bash
cd local_proteins_demo/viz_url
python -m http.server 8000      # then open http://localhost:8000
```

A simple `open index.html` (macOS) or double-click won't work for URL mode
because browsers block `file://` `fetch()` calls for security reasons. The
`http.server` step is one line and only needed for URL mode.

Pin a point and the card fetches `pdbs/<id>.pdb` and feeds it to 3Dmol.js.


## Inline mode (self-contained single file)

Same data, different transport. The PDB text is gzip+base64-embedded so the
single `index.html` is fully portable — you can email it, drop it in a Slack,
upload to a static host, no sidecar needed.

The trade-off is file size: each PDB shows up in the bundle, even ones the user
never pins. Practical limit is a few hundred small structures.


In [ ]:
pdb_texts = [Path(p).read_text() for p in paths]

OUT_TEXT = PROJECT_DIR / "viz_text"
OUT_TEXT.mkdir(exist_ok=True)

viz_text = TmapViz()
viz_text.title = "Local proteins — text mode"
viz_text.point_size = 18.0
viz_text.set_points(xy[:, 0], xy[:, 1])
viz_text.add_label("PDB ID", df["id"].tolist())
viz_text.add_label("Name", df["name"].tolist())
viz_text.add_3d_structures(pdb_texts, source="text", fmt="pdb")

text_html = OUT_TEXT / "index.html"
viz_text.write_html(text_html)
print(f"index.html: {text_html.stat().st_size / 1024:.0f} KB  (PDBs embedded)")


index.html: 1590 KB  (PDBs embedded)


### When to use which

- **File-backed** for the normal local workflow: pass `Path`s and let TMAP copy
  the sidecar files into the output bundle. Serve the folder with `python -m http.server`.
- **URL** when your structures are already hosted or you want to manage sidecars manually.
  Relative URLs also need a static file server (`python -m http.server`, S3, GitHub Pages).
- **Text** only for tiny demos where embedding raw PDB/CIF text is acceptable.

In both cases the rest of the pipeline — colors, layouts, tooltips, search,
filters — works exactly the same; you swap one method call.


## Recap

```python
from tmap.visualization import TmapViz

viz = TmapViz()
viz.set_points(x, y)
viz.add_label("name", names)

# Pick one of:
viz.add_3d_structure_files(paths)                              # local files, copied sidecars
viz.add_3d_structures(pdb_urls)                                # already-hosted URLs
viz.add_3d_structures(pdb_texts, source="text")               # raw text payload

viz.write_static("out/")
```

Whatever you can write to disk as a PDB or mmCIF file, this renders. No
AlphaFold required.
